In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw" / "ml-latest-small"
processed_dir = project_root / "data" / "processed"

movies = pd.read_csv(raw_dir / "movies.csv")
ratings = pd.read_csv(raw_dir / "ratings.csv")

ratings["rated_at"] = pd.to_datetime(
    ratings["timestamp"], unit="s", utc=True
)

train_end = pd.Timestamp("2009-11-17 01:21:52+00:00")
validation_end = pd.Timestamp("2016-03-22 08:26:11+00:00")

train_ratings = ratings.loc[
    ratings["rated_at"] <= train_end
].copy()

validation_ratings = ratings.loc[
    (ratings["rated_at"] > train_end)
    & (ratings["rated_at"] <= validation_end)
].copy()

assert len(train_ratings) == 60502
assert len(validation_ratings) == 20167

In [6]:
training_feedback = train_ratings.copy()
training_feedback["liked"] = (
    training_feedback["rating"] >= 4.0
).astype(int)

user_movie_matrix = (
    training_feedback
    .pivot(index="userId", columns="movieId", values="liked")
    .fillna(0)
    .astype(int)
)

movie_user_matrix = user_movie_matrix.T

movie_popularity = (
    train_ratings
    .groupby("movieId")
    .size()
    .reset_index(name="training_rating_count")
    .merge(
        movies[["movieId", "title"]],
        on="movieId",
        validate="one_to_one"
    )
    .sort_values(
        ["training_rating_count", "movieId"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

In [7]:
training_plot_catalog = pd.read_csv(
    processed_dir / "training_plot_catalog.csv"
)

assert training_plot_catalog["movieId"].is_unique
assert set(training_plot_catalog["movieId"]) == set(
    train_ratings["movieId"]
)

training_plot_catalog["has_plot"] = (
    training_plot_catalog["plot"]
    .fillna("")
    .str.strip()
    .ne("")
)

text_movies = (
    training_plot_catalog.loc[training_plot_catalog["has_plot"]]
    .sort_values("movieId")
    .reset_index(drop=True)
    .copy()
)

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    norm="l2"
)

plot_tfidf = tfidf_vectorizer.fit_transform(text_movies["plot"])

print("Training ratings:", len(train_ratings))
print("Validation ratings:", len(validation_ratings))
print("CF matrix shape:", movie_user_matrix.shape)
print("Movies with plots:", len(text_movies))
print("TF-IDF matrix shape:", plot_tfidf.shape)

Training ratings: 60502
Validation ratings: 20167
CF matrix shape: (6233, 381)
Movies with plots: 5036
TF-IDF matrix shape: (5036, 31609)
